
```
███╗   ███╗██╗███╗   ██╗███████╗ ██████╗ ██████╗ ██╗      █████╗ ██████╗
████╗ ████║██║████╗  ██║██╔════╝██╔════╝██╔═══██╗██║     ██╔══██╗██╔══██╗
██╔████╔██║██║██╔██╗ ██║█████╗  ██║     ██║   ██║██║     ███████║██████╔╝
██║╚██╔╝██║██║██║╚██╗██║██╔══╝  ██║     ██║   ██║██║     ██╔══██║██╔══██╗
██║ ╚═╝ ██║██║██║ ╚████║███████╗╚██████╗╚██████╔╝███████╗██║  ██║██████╔╝
╚═╝     ╚═╝╚═╝╚═╝  ╚═══╝╚══════╝ ╚═════╝ ╚═════╝ ╚══════╝╚═╝  ╚═╝╚═════╝
```
**Run a Minecraft Server on Google Colab!**

---

The script below will run your server. You'll have to create a server first to be able to use it - don't worry, the scripts below will do the majority of the work for you. You might also want to change the default region to your region, check below.

In [17]:
import os
import re
import json

# Update the package lists
!sudo apt update &>/dev/null && echo "apt cache successfully updated" || echo "apt cache update failed, you might receive stale packages"
# Install OpenJDK 17
# !wget -qO - https://adoptopenjdk.jfrog.io/adoptopenjdk/api/gpg/key/public | sudo apt-key add -
# !sudo add-apt-repository --yes https://adoptopenjdk.jfrog.io/adoptopenjdk/deb/ &>/dev/null || echo "Failed to add repo. Still can be ignored if openjdk17 gets installed."
!sudo apt-get install openjdk-17-jre-headless &>/dev/null && echo "Yay! Openjdk17 has been successfully installed." || echo "Failed to install OpenJdk17."
…  !./cloudflared-linux-amd64 tunnel --url tcp://127.0.0.1:25565 & java $memory_allocation $server_flags -jar $jar_name nogui


Java version:
openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working dir: /content/drive/My Drive/Minecraft-server
banned-ips.json		   ngrok.log
banned-players.json	   ngrok-v3-stable-linux-amd64.tgz
bukkit.yml		   ngrok-v3-stable-linux-amd64.tgz.1
cache			   ops.json
cloudflared-linux-amd64    permissions.yml
cloudflared-linux-amd64.1  plugins
cloudflared-linux-amd64.2  server.jar
cloudflared-linux-amd64.3  server.properties
colabconfig.json	   spigot.yml
commands.yml		   usercache.json
config			   version_history.json
eula.txt		   versions
help.yml		   whitelist.json
libraries		   world
logs			   world_nether
ngrok			   world_the_end
Installing ngrok...
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Starting

KeyboardInterrupt: 

In [21]:
import os
import re
import json
import time
import subprocess
from google.colab import drive

# =========================
# 1. UPDATE + JAVA 17
# =========================
os.system("sudo apt update -y > /dev/null 2>&1")
os.system("sudo apt install -y openjdk-17-jre-headless > /dev/null 2>&1")

print("Java version:")
os.system("java -version")

# =========================
# 2. MOUNT GOOGLE DRIVE
# =========================
drive.mount('/content/drive')

SERVER_DIR = "/content/drive/My Drive/Minecraft-server"
os.makedirs(SERVER_DIR, exist_ok=True)
os.chdir(SERVER_DIR)
print("Working dir:", os.getcwd())
os.system("ls")

# =========================
# 3. SERVER SETTINGS
# =========================
jar_name = "server.jar"

memory_allocation = "-Xmx6G -Xms6G"
server_flags = (
    "-XX:+UseG1GC -XX:+ParallelRefProcEnabled "
    "-XX:MaxGCPauseMillis=200 -XX:+UnlockExperimentalVMOptions "
    "-XX:+DisableExplicitGC -XX:+AlwaysPreTouch "
    "-XX:G1NewSizePercent=30 -XX:G1MaxNewSizePercent=40 "
    "-XX:G1HeapRegionSize=8M -XX:G1ReservePercent=20 "
    "-XX:G1HeapWastePercent=5 -XX:G1MixedGCCountTarget=4 "
    "-XX:InitiatingHeapOccupancyPercent=15 "
    "-XX:G1MixedGCLiveThresholdPercent=90 "
    "-XX:G1RSetUpdatingPauseTimePercent=5 "
    "-XX:SurvivorRatio=32 -XX:+PerfDisableSharedMem "
    "-XX:MaxTenuringThreshold=1"
)

# =========================
# 4. NGROK TCP
# =========================
print("Installing ngrok...")
os.system("wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz")
os.system("tar -xzf ngrok-v3-stable-linux-amd64.tgz")
os.system("chmod +x ngrok")

NGROK_TOKEN = "37o1P5E9bvAcQU9P0UXpOlntdoJ_7Y6u5odgRfDPadq62JppE"
os.system(f"./ngrok config add-authtoken {NGROK_TOKEN}")

print("Starting ngrok TCP tunnel...")

# Запуск ngrok через subprocess и чтение API
ngrok_proc = subprocess.Popen(["./ngrok", "tcp", "25565"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Ждём пока ngrok выдаст адрес
time.sleep(5)  # Даем ngrok время запуститься

# Используем API ngrok для получения адреса
import requests

def get_ngrok_address():
    try:
        tunnels = requests.get("http://127.0.0.1:4040/api/tunnels").json()
        for t in tunnels["tunnels"]:
            if t["proto"] == "tcp":
                return t["public_url"].replace("tcp://", "")
    except:
        return None

mc_address = None
for i in range(10):
    mc_address = get_ngrok_address()
    if mc_address:
        break
    time.sleep(1)

if not mc_address:
    raise RuntimeError("❌ Не удалось получить ngrok адрес")

print("======================================")
print("🎮 MINECRAFT SERVER ADDRESS:")
print(mc_address)
print("======================================")

# =========================
# 5. START MINECRAFT SERVER
# =========================
print("Starting Minecraft server...")
os.system(f"java {memory_allocation} {server_flags} -jar {jar_name} nogui")


Java version:
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working dir: /content/drive/My Drive/Minecraft-server
Installing ngrok...
Starting ngrok TCP tunnel...
🎮 MINECRAFT SERVER ADDRESS:
0.tcp.us-cal-1.ngrok.io:14565
Starting Minecraft server...


2

# Make-a-Server

The code below will download a server for you and accept the EULA. After running these scripts, your server will be ready to run.

**Download the Minecraft server**

The code below will download Paper, a high-performance fork of the Vanilla server.
Other server platforms can be used by placing the server.jar in the Drive folder manually.

In [11]:
version = '1.19.2'
server_type = 'paper'

from google.colab import drive
import requests
import json
import os

drive.mount('/content/drive')

server_dir = "/content/drive/My Drive/Minecraft-server"
os.makedirs(server_dir, exist_ok=True)
os.chdir(server_dir)

if server_type == "paper":
    base = "https://api.papermc.io/v2/projects/paper"

    a = requests.get(f"{base}/versions/{version}")
    a.raise_for_status()
    latest_build = a.json()["builds"][-1]

    b = requests.get(f"{base}/versions/{version}/builds/{latest_build}")
    b.raise_for_status()
    jar_name = b.json()["downloads"]["application"]["name"]

    url = f"{base}/versions/{version}/builds/{latest_build}/downloads/{jar_name}"

print("Downloading:", url)

r = requests.get(url)
r.raise_for_status()

with open("server.jar", "wb") as f:
    f.write(r.content)

json.dump({"server_type": server_type}, open("colabconfig.json", "w"))

print("Done!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Downloading: https://api.papermc.io/v2/projects/paper/versions/1.19.2/builds/307/downloads/paper-1.19.2-307.jar
Done!


**Automatically accept the EULA**

In [10]:
# Please read the file stored in your server folder before running this command.
# Also, go to https://www.minecraft.net/en-us/eula to read Minecraft's EULA.

# Make sure Drive is mounted
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/My Drive/Minecraft-server"
!echo "eula=true" >> eula.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/My Drive/Minecraft-server


# Debug
SSH access to host OS - Thanks to [colab-ssh](https://github.com/WassimBenzarti/colab-ssh).

<p style="color:red;">YOU MIGHT GET BANNED</p>

In [ ]:
#@title Colab-ssh tunnel
#@markdown Execute this cell to open the ssh tunnel. Check [colab-ssh documentation](https://github.com/WassimBenzarti/colab-ssh) for more details.

# Install colab_ssh on google colab
!pip install colab_ssh --upgrade

from colab_ssh import launch_ssh_cloudflared, init_git_cloudflared
ssh_tunnel_password = "<PUT_YOUR_PASSWORD_HERE>" #@param {type: "string"}
launch_ssh_cloudflared(password=ssh_tunnel_password)

In [ ]:
#Get public address (ngrok)
! curl -s http://localhost:4040/api/tunnels | python3 -c \
    "import sys, json; print(json.load(sys.stdin)['tunnels'][0]['public_url'])"

In [ ]:
## For inspecting the minecraft server directory ##
%cd "/content/drive/My Drive/Minecraft-server"
!ls
